In [5]:
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


In [6]:
%pip install transformers sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install --upgrade accelerate
%pip uninstall -y transformers accelerate

Note: you may need to restart the kernel to use updated packages.
Found existing installation: transformers 5.5.0
Uninstalling transformers-5.5.0:
  Successfully uninstalled transformers-5.5.0
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install transformers accelerate

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
Note: you may need to restart the kernel to use updated packages.


In [34]:
from transformers import pipeline, set_seed,AutoModelForSeq2SeqLM,AutoTokenizer
from datasets import load_dataset, load_from_disk
# import matplotlib.pyplot as plt
# from datasets import load_dataset,load_metric
# import nltk
# from nltk.tokenize import sent_tokenize

# from tqdm import tqdm
import torch

# nltk.download("punkt")

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [11]:
modelCkpt = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(modelCkpt)

In [12]:
model_pegasus= AutoModelForSeq2SeqLM.from_pretrained(modelCkpt).to(device)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1417.48it/s]


In [13]:
from datasets import load_dataset

# dataset = load_dataset("nyamuda/samsum", split="train[:5%]")
dataset = load_dataset("nyamuda/samsum")
dataset.save_to_disk("sumsum_local")

Saving the dataset (1/1 shards): 100%|██████████| 819/819 [00:00<00:00, 74037.87 examples/s]


In [14]:
# dataset
dataset["train"]["dialogue"][0]

# for split in dataset
    


"Amanda: I baked  cookies. Do you want some?\r\nJerry: Sure!\r\nAmanda: I'll bring you tomorrow :-)"

In [15]:
splitLengths=[len(dataset[split]) for split in dataset]
print(f"splt le:{splitLengths}")
# print(f"feature:{dataset["test"][1]}")

splt le:[14732, 818, 819]


In [16]:
def convertExamplesToFeatures(example):
    target_text = example["summary"]

    model_inputs = tokenizer(
        example["dialogue"],
        max_length=512,
        truncation=True
    )
    
    # with tokenizer.as_target_tokenizer(): #as_target_tokenizer() is deprecated (old method)
    labels = tokenizer(
            target_text,
            max_length=128,
            truncation=True,
            target_text=target_text 
        )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [17]:
dataset_tokenized = dataset.map(convertExamplesToFeatures,batched=True)

In [18]:
dataset_tokenized

DatasetDict({
    train: Dataset({
        features: ['id', 'summary', 'dialogue', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14732
    })
    validation: Dataset({
        features: ['id', 'summary', 'dialogue', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'summary', 'dialogue', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

In [19]:
dataset_tokenized["train"]["input_ids"][1]

[25051,
 10,
 2645,
 33,
 25,
 10601,
 21,
 16,
 48,
 4356,
 58,
 15865,
 10,
 18587,
 7,
 38,
 373,
 5,
 25051,
 10,
 1212,
 396,
 1603,
 15865,
 10,
 1651,
 1]

In [20]:
dataset_tokenized["train"]["attention_mask"][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [21]:
tokenizer.decode(dataset_tokenized["train"]["input_ids"][1])

'Olivia: Who are you voting for in this election? Oliver: Liberals as always. Olivia: Me too!! Oliver: Great</s>'

In [22]:
tokenizer.decode(dataset_tokenized["train"]["labels"][1])

'Olivia and Olivier are voting for liberals in this election.</s>'

In [23]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, 
    model=modelCkpt,        # Providing the model helps handle decoder_input_ids
    padding=True,       # Dynamic padding to the longest sequence in the batch
    return_tensors="pt" # Returns PyTorch tensors
)

In [24]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
    output_dir="./results",
    use_cpu=True,                  # Force CPU usage
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4,  # Lowered slightly for speed
    max_steps=20,                  # Short run just to "see" it work
    fp16=False,
    logging_steps=1,               # See progress every single step
    eval_strategy="steps", 
    eval_steps=200,    # Skip eval to save time
    save_steps=100                 # Don't waste time saving big files yet
)

In [25]:

trainer = Trainer(
    model=model_pegasus,                         # Your model (e.g., T5, BART, or GPT2)
    args=training_args,                  # The arguments we just defined
    train_dataset=dataset_tokenized["train"], 
    eval_dataset=dataset_tokenized["test"],   # Optional: used for evaluation
    data_collator=data_collator          # The Seq2Seq collator we imported earlier
)

In [26]:
trainer.train()

Step,Training Loss,Validation Loss
20,11.976548,2.580478


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


TrainOutput(global_step=20, training_loss=11.832731533050538, metrics={'train_runtime': 366.1639, 'train_samples_per_second': 0.218, 'train_steps_per_second': 0.055, 'total_flos': 2923065704448.0, 'train_loss': 11.832731533050538, 'epoch': 0.005430355688297583})

In [49]:
def yield_dataset_chunks(list_of_elements, batch_size):
    """
    Yield successive batch-sized chunks from a list.
    Works for Python lists, columns, and arrays!
    """
    num_rows = len(list_of_elements)
    for i in range(0, num_rows, batch_size):
        # FIX: Use standard Python slicing [start:end] instead of .select()
        batch = list_of_elements[i : i + batch_size]
        yield batch

In [50]:
from tqdm import tqdm

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer, 
                                batch_size=2, device=device, column_text="dialogue", 
                                column_summary="summary"):
    
    # 1. Don't use list(). Keep it as a generator to save RAM!
    article_batches = yield_dataset_chunks(dataset[column_text], batch_size)
    target_batches = yield_dataset_chunks(dataset[column_summary], batch_size)

    # 2. Add a Progress Bar (tqdm) so you don't feel like it's frozen
    for article_batch, target_batch in tqdm(zip(article_batches, target_batches), total=len(dataset)//batch_size):
        
        # 3. Tokenize inputs (Truncation is KEY for CPU speed)
        inputs = tokenizer(article_batch, max_length=128, truncation=True, 
                          padding="max_length", return_tensors="pt").to(device)
        
        # 4. Generate Summaries
        # max_new_tokens=30 keeps the CPU from looping forever on long text
        summaries = model.generate(input_ids=inputs["input_ids"], 
                                   attention_mask=inputs["attention_mask"],
                                   max_new_tokens=30, 
                                   num_beams=1) # num_beams=1 is much faster on CPU
        
        # 5. Decode and Store
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True, 
                             clean_up_tokenization_spaces=True) for s in summaries]
        
        # 6. Add results to the metric
        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # 7. Final Score
    return metric.compute()

In [ ]:
pip install evaluate rouge_score

In [51]:
import evaluate

# 1. Define the names you want to extract later
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

# 2. Load the metric
metric = evaluate.load("rouge")

In [54]:
print(type(dataset["test"]))
# 1. Run the evaluation on 10 rows
# Use .select() on the 'test' split ITSELF, not a column inside it
small_test_dataset = dataset["test"].select(range(10))

# Now pass this small "table" to your function
scr = calculate_metric_on_test_ds(
    dataset=small_test_dataset, 
    metric=metric,
    model=trainer.model, 
    tokenizer=tokenizer, 
    batch_size=1, 
    # device=device,
    column_text="dialogue", 
    column_summary="summary"
)

# 2. Print the results
print(scr)
# rgDict=dict((rn,scr[rn].mid.fmeasre)for rn in rouge_names)
# pd.datafr(rgDict,index=[])

<class 'datasets.arrow_dataset.Dataset'>


100%|██████████| 10/10 [00:04<00:00,  2.25it/s]

{'rouge1': np.float64(0.21843914284304572), 'rouge2': np.float64(0.05114790591606937), 'rougeL': np.float64(0.17949715990001092), 'rougeLsum': np.float64(0.1798073053527)}


In [ ]:
# Or the even cleaner Python way (Dictionary Comprehension):
rgDict = {rn: scr[rn] for rn in rouge_names}
print(rgDict)
# pd.datafr(rgDict,index=[])

{'rouge1': np.float64(0.21843914284304572), 'rouge2': np.float64(0.05114790591606937), 'rougeL': np.float64(0.17949715990001092), 'rougeLsum': np.float64(0.1798073053527)}


In [57]:
import pandas as pd
# We pass the dictionary and give the row a label (the index)
df = pd.DataFrame(rgDict, index=["t5-small"])
print(df)

            rouge1    rouge2    rougeL  rougeLsum
t5-small  0.218439  0.051148  0.179497   0.179807


In [59]:
# Define your folder name
directory = "./a_t5_summarizer"

# 1. Save the "Brain" (Weights and Config)
model_pegasus.save_pretrained(directory)

# 2. Save the "Dictionary" (Vocabulary and Rules)
tokenizer.save_pretrained(directory)

print(f"Model and Tokenizer successfully saved to {directory}!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.44it/s]

Model and Tokenizer successfully saved to ./a_t5_summarizer!


In [62]:
#load
tokenizer = AutoTokenizer.from_pretrained(directory)

In [71]:
# 1. Define your "Recipe" for generation
gen_kwargs = {
    "length_penalty": 0.8,    # Keep it concise for a chat summary
    "num_beams": 4,           # Look at 4 possibilities (good for T5)
    "max_length": 128,        # Don't let the CPU run forever
    "no_repeat_ngram_size": 3 # Stop the AI from saying "and then, and then..."
}

# sampleText=dataset["test"][0]["dialogue"]
# refernce=dataset["test"][0]["summary"]
# pipe=pipeline("summarixation",model="t5-small",tokenizer=tokenizer)
# The ** unpacks the dictionary into the function
output = model_pegasus.generate(input_ids=inputs["input_ids"], **gen_kwargs)

In [69]:

# 1. Grab the "Question" and the "Answer Key"
sample_text = dataset["test"][0]["dialogue"]
reference = dataset["test"][0]["summary"]

# 2. Create the "Easy Mode" inference object
# Fix: changed "summarixation" to "summarization"
# This works regardless of the 'task' name because it talks to the model directly
inputs = tokenizer(sample_text, return_tensors="pt").to(device)
outputs = model_pegasus.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
# pipe = pipeline("text-generation", model=model_pegasus, tokenizer=tokenizer)

Hannah: Hey, do you have Betty's number? Amanda: Lemme check Hannah:


In [ ]:
# print("\n--- HUMAN SUMMARY ---")
print(reference)
#Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
# print(sample_text)

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.
